# Predicting Professional Tennis Match Outcomes with Machine Learning

## 1. Settings

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 50)

START_YEAR = 2010
END_YEAR = 2024

DATA_URL = "https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/atp_matches_{year}.csv"

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 2. Download and Load the Data

This downloads the yearly ATP files. If the file already exists locally, the notebook uses the local copy.

In [ ]:
all_years = []

for year in range(START_YEAR, END_YEAR + 1):
    local_file = RAW_DIR / f"atp_matches_{year}.csv"
    url = DATA_URL.format(year=year)

    if local_file.exists():
        year_data = pd.read_csv(local_file)
    else:
        year_data = pd.read_csv(url)
        year_data.to_csv(local_file, index=False)

    year_data["source_year"] = year
    all_years.append(year_data)

matches = pd.concat(all_years, ignore_index=True)

print(matches.shape)
matches.head()

## 3. Keep the Columns We Need

We are keeping basic match information and pre-match information like ranking, ranking points, and age. We are not using same-match serving stats yet because those would not be known before predicting a match.

In [ ]:
columns_to_keep = [
    "tourney_id",
    "tourney_name",
    "surface",
    "tourney_date",
    "match_num",
    "best_of",
    "round",
    "winner_id",
    "winner_name",
    "winner_age",
    "loser_id",
    "loser_name",
    "loser_age",
    "winner_rank",
    "winner_rank_points",
    "loser_rank",
    "loser_rank_points",
    "source_year",
]

matches = matches[columns_to_keep].copy()

matches.head()

## 4. Basic Cleaning

In [ ]:
matches["match_date"] = pd.to_datetime(
    matches["tourney_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

number_columns = [
    "winner_age",
    "loser_age",
    "winner_rank",
    "loser_rank",
    "winner_rank_points",
    "loser_rank_points",
]

for column in number_columns:
    matches[column] = pd.to_numeric(matches[column], errors="coerce")

cleaned = matches.dropna(
    subset=["match_date", "winner_id", "loser_id", "winner_rank", "loser_rank"]
).copy()

print("Original rows:", len(matches))
print("Rows after basic cleaning:", len(cleaned))
cleaned.head()

## 5. Starter Feature Engineering

These features are simple and easy to explain. Since the data is still in winner/loser format, these are mostly useful for early exploration and for building a ranking baseline later.

In [ ]:
cleaned["ranking_diff"] = cleaned["winner_rank"] - cleaned["loser_rank"]
cleaned["rank_points_diff"] = cleaned["winner_rank_points"] - cleaned["loser_rank_points"]
cleaned["age_diff"] = cleaned["winner_age"] - cleaned["loser_age"]
cleaned["higher_ranked_won"] = (cleaned["winner_rank"] < cleaned["loser_rank"]).astype(int)
cleaned["match_year"] = cleaned["match_date"].dt.year

cleaned[[
    "match_date",
    "surface",
    "winner_name",
    "loser_name",
    "winner_rank",
    "loser_rank",
    "ranking_diff",
    "higher_ranked_won",
]].head()

## 6. Quick Checks

In [ ]:
cleaned.isna().mean().sort_values(ascending=False).head(10)

In [ ]:
cleaned.groupby("match_year").size().tail()

In [ ]:
cleaned["surface"].value_counts(dropna=False)

In [ ]:
higher_ranked_win_rate = cleaned["higher_ranked_won"].mean()
print(f"Higher-ranked player won {higher_ranked_win_rate:.1%} of matches.")

## 7. Simple Visualization

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(cleaned["ranking_diff"], bins=50)
plt.title("Ranking Difference: Winner Rank - Loser Rank")
plt.xlabel("Ranking Difference")
plt.ylabel("Number of Matches")
plt.show()

## 8. Save Processed Files

In [ ]:
combined_path = PROCESSED_DIR / "atp_matches_combined.csv"
cleaned_path = PROCESSED_DIR / "atp_matches_cleaned_starter.csv"

matches.to_csv(combined_path, index=False)
cleaned.to_csv(cleaned_path, index=False)

print("Saved:", combined_path)
print("Saved:", cleaned_path)

## 9. Modeling Goal and Evaluation Plan

We want to see if pre-match factors can predict the winner better than a ranking-only rule. The original data is stored from the winner's perspective, so the modeling data below rewrites each match as Player 1 vs. Player 2 and randomly flips which real player is Player 1. That keeps the target honest: "player_1_won" is 1 when Player 1 won and 0 when Player 2 won.

We compare every model to a baseline that always picks the higher-ranked player. Cross-validation is time-aware, so each validation fold is later in the tennis calendar than its training fold.

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, make_scorer, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 211
N_SPLITS = 5
RECENT_WINDOW = 10

## 10. Build Leakage-Safe Pre-Match Features

These features use only information that would have been available before a match started: rankings, ranking points, ages, surface, round, match format, and each player's prior win rates. Same-match serve and return statistics are intentionally excluded because those are only known after the match.

In [ ]:
def add_player_history_features(match_data, recent_window=10):
    ordered = match_data.sort_values(
        ["match_date", "tourney_id", "match_num"],
        kind="mergesort"
    ).reset_index(drop=True).copy()
    ordered["match_id"] = ordered.index

    winner_rows = ordered[[
        "match_id", "match_date", "tourney_id", "match_num", "surface",
        "winner_id", "winner_rank", "winner_rank_points", "winner_age"
    ]].rename(columns={
        "winner_id": "player_id",
        "winner_rank": "player_rank",
        "winner_rank_points": "player_rank_points",
        "winner_age": "player_age",
    })
    winner_rows["won"] = 1

    loser_rows = ordered[[
        "match_id", "match_date", "tourney_id", "match_num", "surface",
        "loser_id", "loser_rank", "loser_rank_points", "loser_age"
    ]].rename(columns={
        "loser_id": "player_id",
        "loser_rank": "player_rank",
        "loser_rank_points": "player_rank_points",
        "loser_age": "player_age",
    })
    loser_rows["won"] = 0

    player_rows = pd.concat([winner_rows, loser_rows], ignore_index=True)
    player_rows = player_rows.sort_values(
        ["match_date", "tourney_id", "match_num", "match_id", "won"],
        kind="mergesort"
    ).reset_index(drop=True)

    player_group = player_rows.groupby("player_id", sort=False)
    player_rows["matches_before"] = player_group.cumcount()
    player_rows["wins_before"] = player_group["won"].cumsum() - player_rows["won"]
    player_rows["career_win_rate_before"] = (
        player_rows["wins_before"] / player_rows["matches_before"]
    ).fillna(0.5)

    player_rows["recent_wins_before"] = player_group["won"].transform(
        lambda values: values.shift().rolling(recent_window, min_periods=1).sum()
    )
    player_rows["recent_matches_before"] = player_group["won"].transform(
        lambda values: values.shift().rolling(recent_window, min_periods=1).count()
    )
    player_rows["recent_win_rate_before"] = (
        player_rows["recent_wins_before"] / player_rows["recent_matches_before"]
    ).fillna(0.5)

    surface_group = player_rows.groupby(["player_id", "surface"], sort=False)
    player_rows["surface_matches_before"] = surface_group.cumcount()
    player_rows["surface_wins_before"] = surface_group["won"].cumsum() - player_rows["won"]
    player_rows["surface_win_rate_before"] = (
        player_rows["surface_wins_before"] / player_rows["surface_matches_before"]
    ).fillna(0.5)

    return ordered, player_rows


ordered_matches, player_history = add_player_history_features(cleaned, RECENT_WINDOW)
player_history.head()

In [ ]:
def make_modeling_table(match_data, player_history, random_state=RANDOM_STATE):
    rng = pd.Series(
        np.random.default_rng(random_state).integers(0, 2, size=len(match_data)),
        index=match_data["match_id"].to_numpy(),
        name="flip_players"
    )

    player_features = [
        "match_id",
        "player_id",
        "player_rank",
        "player_rank_points",
        "player_age",
        "matches_before",
        "career_win_rate_before",
        "recent_win_rate_before",
        "surface_win_rate_before",
        "won",
    ]

    winners = player_history.loc[player_history["won"] == 1, player_features].set_index("match_id")
    losers = player_history.loc[player_history["won"] == 0, player_features].set_index("match_id")

    rows = match_data.set_index("match_id")[[
        "match_date", "tourney_name", "surface", "round", "best_of", "winner_name", "loser_name"
    ]].copy()
    rows["flip_players"] = rng

    for column in player_features:
        if column == "match_id":
            continue
        rows[f"winner_{column}"] = winners[column]
        rows[f"loser_{column}"] = losers[column]

    player_columns = [
        "player_id",
        "player_rank",
        "player_rank_points",
        "player_age",
        "matches_before",
        "career_win_rate_before",
        "recent_win_rate_before",
        "surface_win_rate_before",
    ]

    for column in player_columns:
        rows[f"p1_{column}"] = np.where(
            rows["flip_players"].eq(0),
            rows[f"winner_{column}"],
            rows[f"loser_{column}"]
        )
        rows[f"p2_{column}"] = np.where(
            rows["flip_players"].eq(0),
            rows[f"loser_{column}"],
            rows[f"winner_{column}"]
        )

    rows["player_1_won"] = rows["flip_players"].eq(0).astype(int)
    rows["match_year"] = rows["match_date"].dt.year

    rows["rank_diff"] = rows["p1_player_rank"] - rows["p2_player_rank"]
    rows["rank_points_diff"] = rows["p1_player_rank_points"] - rows["p2_player_rank_points"]
    rows["age_diff"] = rows["p1_player_age"] - rows["p2_player_age"]
    rows["experience_diff"] = rows["p1_matches_before"] - rows["p2_matches_before"]
    rows["career_win_rate_diff"] = rows["p1_career_win_rate_before"] - rows["p2_career_win_rate_before"]
    rows["recent_win_rate_diff"] = rows["p1_recent_win_rate_before"] - rows["p2_recent_win_rate_before"]
    rows["surface_win_rate_diff"] = rows["p1_surface_win_rate_before"] - rows["p2_surface_win_rate_before"]

    return rows.reset_index()


modeling = make_modeling_table(ordered_matches, player_history)

modeling[[
    "match_date", "surface", "round", "winner_name", "loser_name",
    "flip_players", "rank_diff", "recent_win_rate_diff", "player_1_won"
]].head()

## 11. Cross-Validation Setup

The validation uses "TimeSeriesSplit" to train on earlier matches and test on later matches. This is stricter than random folds and better matches the real prediction task, where we would use the past to predict future matches.

In [ ]:
numeric_features = [
    "rank_diff",
    "rank_points_diff",
    "age_diff",
    "experience_diff",
    "career_win_rate_diff",
    "recent_win_rate_diff",
    "surface_win_rate_diff",
    "p1_player_rank",
    "p2_player_rank",
    "p1_player_rank_points",
    "p2_player_rank_points",
    "p1_player_age",
    "p2_player_age",
    "p1_matches_before",
    "p2_matches_before",
    "match_year",
]

categorical_features = ["surface", "round", "best_of"]
feature_columns = numeric_features + categorical_features

modeling = modeling.sort_values("match_date", kind="mergesort").reset_index(drop=True)
X = modeling[feature_columns].copy()
y = modeling["player_1_won"].astype(int).copy()

cv = TimeSeriesSplit(n_splits=N_SPLITS)

print("Modeling rows:", len(modeling))
print("Target balance:")
print(y.value_counts(normalize=True).rename("share"))